# Training a simple model using Lightning library
Inspired by: https://github.com/podondra/downscaling/

In [12]:
import torch
import xarray as xr
import rasterio
import cartopy
import lightning
import torchmetrics
from lightning.pytorch.utilities.types import OptimizerLRScheduler


In [16]:
class AbstractModel(lightning.LightningModule):
    def __init__(self):
        super().__init__()
        self.save_hyperparameters()
        self.train_metrics = torchmetrics.MetricCollection(
            {
                "mae" : torchmetrics.MeanAbsoluteError(),
                "mse" : torchmetrics.MeanSquaredError(),
                "rmse" : torchmetrics.MeanSquaredError(squared=False),
            },
            prefix = 'train/',
        )
        self.val_metrics = self.train_metrics.clone(prefix='val/')

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss(y_hat, y)
        self.train_metrics(y_hat, y)
        self.log_dict(self.train_metrics, on_step=True, on_epoch=False)
        self.log("train/loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss(y_hat, y)
        self.val_metrics(y_hat, y)
        self.log_dict(self.val_metrics, on_step=False, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        return self(x)

    def on_validation_epoch_end(self):
        metrics = self.val_metrics.compute()
        self.log('hp_metric', metrics['val/rmse'])

In [17]:
class SimpleCNN(AbstractModel):
    def __init__(self):
        super().__init__()
        self.loss = torch.nn.MSELoss()
        self.net = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(1, 1, 61),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(1, 1, 61),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(1, 1, 61),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(1, 1, 61),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(1, 1, 61),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(1, 1, 61)
        )

    def forward(self, x):
        return self.net(x)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [18]:
class ReKIS(torch.utils.data.Dataset):
    """ReKIS dataset"""
    def __init__(self, Y):
        super().__init__()
        # ordinary crop
        Y = Y.isel(easting = slice(0, 400), northing = slice(0, 400))
        # axes labels disappear after crop
        Y.rio.set_spatial_dims('easting', 'northing')

        # obtain train data -> upscaling
        X = Y.rio.reproject(Y.rio.crs, resolution=(10_000, 10_000), resampling= rasterio.enums.Resampling.cubic_spline)

        # conversion to tensors
        self.X = torch.from_numpy(X.values).unsqueeze(1)
        self.Y = torch.from_numpy(Y.values).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


class ReKISDataModule(lightning.LightningDataModule):
    """preserves loaders"""
    def __init__(self, batch_size, path):
        super().__init__()
        self.batch_size = batch_size
        self.path = path

    def setup(self, stage):
        """makes ReKIS datasets from path"""
        Y = xr.open_mfdataset(self.path, decode_coords='all')
        Y = Y['TM']

        # split into train/val sets and make datasets
        self.train = ReKIS(Y.sel(time=slice('1961', '1961')))
        self.val = ReKIS(Y.sel(time=slice('1962', '1962')))

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.train, batch_size = self.batch_size, shuffle = True)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.val, batch_size = self.batch_size)

In [19]:
lightning.seed_everything(42, workers=True)
rekis_module = ReKISDataModule(32, f'../../rci_data/climate/ReKIS/KlimRefDS_v3.1_1961-2023/Raster/Tag/GK4/TM/*.nc')
rekis_module.setup(None)

train_loader = rekis_module.train_dataloader()
val_loader = rekis_module.val_dataloader()

Seed set to 42


In [21]:
trainer = lightning.Trainer(accelerator='gpu', devices=1, max_epochs=5, log_every_n_steps=1)
trainer.fit(SimpleCNN(), train_loader)

`Trainer.fit` stopped: `max_epochs=5` reached.
